In [1]:
import pandas as pd
from weather_pipeline.datasource import build_data_source

era5 = build_data_source("era5")
times = pd.date_range("2020-08-01", "2020-08-31T18:00", freq="6h")
coarse = era5(list(times), ["t2m"]).sel(variable="t2m", drop=True)

In [3]:
from huggingface_hub import hf_hub_download
from downscale.downscaler import Downscaler
from downscale.station_interp import GridInterpolator

model = Downscaler.load(hf_hub_download("EmmaScharfmann/france-temperature-downscaler", "model.pt"))
interp = GridInterpolator.build(coarse.lat, coarse.lon, model.lat.ravel(), model.lon.ravel())
on_fine = interp.apply(coarse).values.reshape(len(times), *model.lat.shape)
fine = model.predict(on_fine, times.to_numpy(), members=4)

model.pt: reconstructing file:   0%|          |  0.00B / 10.3MB            

model.pt: downloading bytes:           |  0.00B            

In [8]:
import xarray as xr

USE_CACHE = True  # False to download the store once and read it locally

if USE_CACHE:
    cerra = xr.open_zarr("hf://datasets/EmmaScharfmann/cerra-france-t2m-2016-2020/target.zarr")
else:
    from huggingface_hub import snapshot_download
    folder = snapshot_download("EmmaScharfmann/cerra-france-t2m", repo_type="dataset")
    cerra = xr.open_zarr(f"{folder}/target.zarr")

truth = cerra.t2m.sel(time=times).values
for name, field in [("coarse", on_fine), ("downscaled", fine.mean(1))]:
    print(name, np.sqrt(((field - truth) ** 2).mean()).round(3), "K")

coarse 1.47 K
downscaled 1.246 K


In [ ]:
import numpy as np, xarray as xr
from downscale.stations import MeteoFranceHourly, load_published_stations, load_stations
from downscale.station_interp import curvilinear_for

USE_CACHE = True  # False to download the observations from Météo-France instead

if USE_CACHE:
    stations = load_published_stations("EmmaScharfmann/france-stations-2020-08")
else:
    stations, _ = load_stations(MeteoFranceHourly(), "france", times[0], times[-1])

read = curvilinear_for(model.lat, model.lon, stations).apply
obs = stations.t2m.sel(time=times).values
for name, field in [("coarse", on_fine), ("downscaled", fine.mean(1))]:
    at_stations = read(xr.DataArray(field, dims=("time", "y", "x"))).values
    print(name, np.sqrt(np.nanmean((at_stations - obs) ** 2)).round(3), "K")